# Sentence Corpus Inspection

In [ ]:
import os
import pandas as pd

PATH = "/home/tom/data/sentence_corpus.csv"

size_gb = os.path.getsize(PATH) / 1e9
print(f"File size: {size_gb:.2f} GB")

In [ ]:
# Actual row count — reads only the 'country' column (fast, handles quoted newlines)
total = 0
for chunk in pd.read_csv(PATH, usecols=["country"], chunksize=1_000_000):
    total += len(chunk)
print(f"Actual row count: {total:,}")

In [ ]:
# Also count raw newlines so we can see the discrepancy
with open(PATH, "rb") as f:
    raw_lines = sum(1 for _ in f) - 1
print(f"Raw newline count (overcounts quoted fields): {raw_lines:,}")
print(f"Difference (embedded newlines in sentences):  {raw_lines - total:,}")

In [ ]:
# Load a sample to inspect structure
df = pd.read_csv(PATH, nrows=100_000)
print(df.dtypes)
df.head()

In [ ]:
# Row counts by source_dataset_type
type_counts = {}
dataset_counts = {}
country_counts = {}

for chunk in pd.read_csv(PATH, usecols=["source_dataset_type", "source_dataset", "country"], chunksize=1_000_000):
    for col, acc in [("source_dataset_type", type_counts),
                     ("source_dataset",      dataset_counts),
                     ("country",             country_counts)]:
        counts = chunk[col].value_counts()
        for k, v in counts.items():
            acc[k] = acc.get(k, 0) + v

print("=== By source_dataset_type ===")
for k, v in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {k:<25} {v:>15,}")

print("\n=== By source_dataset ===")
for k, v in sorted(dataset_counts.items(), key=lambda x: -x[1]):
    print(f"  {k:<30} {v:>15,}")

print("\n=== By country ===")
for k, v in sorted(country_counts.items(), key=lambda x: -x[1]):
    print(f"  {k:<10} {v:>15,}")

In [ ]:
# Date range per source_dataset (sample-based — reads first 1M rows)
df_sample = pd.read_csv(PATH, usecols=["source_dataset", "date"], nrows=1_000_000)
df_sample["date"] = pd.to_datetime(df_sample["date"], errors="coerce")
df_sample.groupby("source_dataset")["date"].agg(["min", "max", "count"])

In [ ]:
# Sentence length distribution (sample)
df_sample2 = pd.read_csv(PATH, usecols=["sentence", "source_dataset_type"], nrows=500_000)
df_sample2["len"] = df_sample2["sentence"].str.len()
print(df_sample2.groupby("source_dataset_type")["len"].describe().round(1))

In [ ]:
# Check for embedded newlines in sentences (what was overcounting the rows)
has_newline = df_sample2["sentence"].str.contains(r"\n", na=False)
print(f"Sentences with embedded newlines: {has_newline.sum():,} / {len(df_sample2):,} ({has_newline.mean()*100:.2f}%)")

In [ ]:
# Inspect a few random rows
df.sample(10)[["sentence", "date", "speaker", "country", "source_dataset", "source_dataset_type"]]